# Final Model: DistilBERT
**Nova IMS - Text Mining 2025/2026 - Group 31**

This notebook contains our final solution. It explicitly loads the final champion model, classifies the test dataset, and saves the predictions to `outputs/pred_31.csv` as requested by the rubric.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

TEST_CSV_PATH = "data/test.csv"
PRED_CSV_PATH = "outputs/pred_31.csv"


## 1. Run Final Predictions (Smart Dispatcher)

Because our project explored both **Classical Machine Learning pipelines** (e.g., TF-IDF + Logistic Regression/XGBoost) and **Deep Learning models** (e.g., fine-tuned DistilBERT/RoBERTa), we built a "Smart Dispatcher" (`src/autotune.py`) to handle the final inference dynamically.

**How `autotune.py` works:**
1. **Leaderboard Analysis:** It parses `outputs/results.csv` to identify the model configuration with the highest `f1_macro` score.
2. **Dynamic Pipeline Routing:**
   - **If the champion is a Deep Learning model** (like DistilBERT): It imports the specific HuggingFace trainer module (`src/distilbert_trainer.py`), loads the pre-trained weights from our local checkpoint, tokenizes the test text, and runs the PyTorch inference loop.
   - **If the champion is a Classical ML model**: It reconstructs the winning `TfidfVectorizer` and Scikit-Learn classifier, safely refits them on 100% of the training data to maximize learning, and then predicts the test set.
3. **Output Generation:** It saves the final predictions to `outputs/pred_31.csv`.

This ensures our final pipeline is 100% robust and reproducible. If a new experiment yields a better result tomorrow, we don't have to change a single line of code here—the Smart Dispatcher will automatically swap to the new champion.

In [ ]:
!python src/autotune.py
print("\nPredictions successfully saved to:", PRED_CSV_PATH)

## 2. Interactive Results
Let's load the generated predictions file to verify the final test outputs and analyze the predicted sentiment distribution.

In [ ]:
# Load predictions
preds_df = pd.read_csv(PRED_CSV_PATH)

print("First 5 predictions:")
display(preds_df.head())

# Plot the predicted sentiment distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=preds_df, x='label', palette='viridis')
plt.title('Predicted Sentiment Distribution on Test Set')
plt.xlabel('Sentiment Label (0=Bearish, 1=Bullish, 2=Neutral)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()